# Machine Learning Zoomcamp 2026 - Homework 2: Linear Regression

Solución detallada y explicada de la Tarea 2 (Regresión Lineal).

### Objetivos:
- EDA básico y detección de variables con datos faltantes.
- Partición 60% train / 20% val / 20% test con barajado determinista.
- Comparación de imputación de valores faltantes (con 0 vs con la media).
- Regularización $L_2$ (Ridge) y selección del mejor factor $r$.
- Evaluación de estabilidad del modelo mediante análisis de semillas (0-9).
- Entrenamiento y evaluación final en conjunto de test.


## 1. Importación y Carga de Datos

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

data_path = Path("../../data/car_fuel_efficiency_2026.csv")
df = pd.read_csv(data_path)

CAR_COLUMNS = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
]
TARGET = "fuel_efficiency_mpg"

data = df[CAR_COLUMNS + [TARGET]].copy()
data.head()


,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## Q1. Columna con valores faltantes
¿Cuál es la columna de las seleccionadas que contiene valores faltantes?

In [2]:
# Pregunta 1
missing = data[CAR_COLUMNS].isna().sum()
missing_col = missing[missing > 0].index[0]
print(f"Columna con valores faltantes: '{missing_col}' (Total nulos: {missing[missing_col]})")


Columna con valores faltantes: 'horsepower' (Total nulos: 877)


## Q2. Mediana de 'horsepower'
¿Cuál es la mediana (percentil 50%) de la variable `horsepower`?

In [3]:
# Pregunta 2
median_hp = data["horsepower"].median()
print(f"Mediana de horsepower: {median_hp:.0f}")


Mediana de horsepower: 254


## Funciones de Partición y Regresión Lineal

In [4]:
def split_with_lecture_shuffle(dataframe: pd.DataFrame, seed: int):
    n = len(dataframe)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    indices = np.arange(n)
    np.random.RandomState(seed).shuffle(indices)

    df_train = dataframe.iloc[indices[:n_train]].copy()
    df_val = dataframe.iloc[indices[n_train : n_train + n_val]].copy()
    df_test = dataframe.iloc[indices[n_train + n_val :]].copy()

    return df_train, df_val, df_test

def train_linear_regression(X_train: pd.DataFrame, y_train: pd.Series, r: float = 0.0):
    matrix = np.column_stack([np.ones(len(X_train)), X_train.to_numpy(dtype=float)])
    XTX = matrix.T @ matrix
    if r > 0:
        regularizer = r * np.eye(XTX.shape[0])
        regularizer[0, 0] = 0.0
        XTX = XTX + regularizer
    weights = np.linalg.solve(XTX, matrix.T @ y_train.to_numpy(dtype=float))
    return weights

def evaluate_linear_regression(X_val: pd.DataFrame, y_val: pd.Series, weights: np.ndarray) -> float:
    matrix = np.column_stack([np.ones(len(X_val)), X_val.to_numpy(dtype=float)])
    y_pred = matrix @ weights
    return float(np.sqrt(mean_squared_error(y_val, y_pred)))

def compute_rmse(df_train, df_val, fill_value, r=0.0):
    X_train = df_train[CAR_COLUMNS].fillna(fill_value)
    y_train = df_train[TARGET]
    X_val = df_val[CAR_COLUMNS].fillna(fill_value)
    y_val = df_val[TARGET]

    weights = train_linear_regression(X_train, y_train, r=r)
    return evaluate_linear_regression(X_val, y_val, weights)


## Q3. Imputación con 0 vs con la media
¿Qué opción da mejor RMSE en validación? (`With 0`, `With mean`, `Both are equally good`)

In [5]:
# Pregunta 3
train_42, val_42, test_42 = split_with_lecture_shuffle(data, seed=42)

rmse_zero = compute_rmse(train_42, val_42, fill_value=0.0)
mean_hp = train_42["horsepower"].mean()
rmse_mean = compute_rmse(train_42, val_42, fill_value=mean_hp)

print(f"RMSE rellenando con 0:    {rmse_zero:.3f} (exacto: {rmse_zero:.5f})")
print(f"RMSE rellenando con media:{rmse_mean:.3f} (exacto: {rmse_mean:.5f})")

if round(rmse_mean, 3) < round(rmse_zero, 3):
    res_q3 = "With mean"
elif round(rmse_zero, 3) < round(rmse_mean, 3):
    res_q3 = "With 0"
else:
    res_q3 = "Both are equally good"
print(f"Resultado: {res_q3}")


RMSE rellenando con 0:    2.205 (exacto: 2.20529)
RMSE rellenando con media:2.202 (exacto: 2.20182)
Resultado: With mean


## Q4. Regresión lineal con regularización (Ridge)
Probando `r` en `[0, 0.01, 0.1, 1, 5, 10, 100]` con NA rellenado con 0.
¿Cuál `r` da el mejor RMSE (redondeado a 4 decimales)? Si hay empate, elegir el menor `r`.

In [6]:
# Pregunta 4
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
scores_q4 = {}

for r in r_values:
    score = compute_rmse(train_42, val_42, fill_value=0.0, r=r)
    scores_q4[r] = round(score, 4)
    print(f"r = {r:<6}: RMSE = {score:.4f}")

best_r = min(r_values, key=lambda r: (scores_q4[r], r))
print(f"Mejor r: {best_r}")


r = 0     : RMSE = 2.2053
r = 0.01  : RMSE = 2.2053
r = 0.1   : RMSE = 2.2053
r = 1     : RMSE = 2.2053
r = 5     : RMSE = 2.2053
r = 10    : RMSE = 2.2053
r = 100   : RMSE = 2.2053
Mejor r: 0


## Q5. Estabilidad ante diferentes semillas
Evaluamos el modelo con semillas del 0 al 9 y calculamos la desviación estándar (`np.std`) de los RMSEs redondeado a 3 decimales.

In [7]:
# Pregunta 5
seed_scores = []
for s in range(10):
    tr, vl, _ = split_with_lecture_shuffle(data, seed=s)
    sc = compute_rmse(tr, vl, fill_value=0.0, r=0.0)
    seed_scores.append(sc)

std_rmse = np.std(seed_scores)
print(f"Scores por semilla: {[round(s, 4) for s in seed_scores]}")
print(f"Desviación estándar de los RMSEs: {std_rmse:.3f}")


Scores por semilla: [2.2392, 2.2021, 2.1634, 2.2044, 2.2019, 2.2458, 2.2697, 2.1908, 2.2166, 2.207]
Desviación estándar de los RMSEs: 0.029


## Q6. Entrenamiento combinado y evaluación en Test
Usar semilla 9, combinar train + val, entrenar con $r=0.001$, imputar NA con 0 y evaluar RMSE en test.

In [8]:
# Pregunta 6
train_9, val_9, test_9 = split_with_lecture_shuffle(data, seed=9)
combined_train = pd.concat([train_9, val_9])

X_comb = combined_train[CAR_COLUMNS].fillna(0.0)
y_comb = combined_train[TARGET]
X_test = test_9[CAR_COLUMNS].fillna(0.0)
y_test = test_9[TARGET]

weights_final = train_linear_regression(X_comb, y_comb, r=0.001)
final_test_rmse = evaluate_linear_regression(X_test, y_test, weights_final)

print(f"RMSE en test dataset: {final_test_rmse:.3f}")


RMSE en test dataset: 2.236


### Resumen de Respuestas HW2:
- **Q1**: `'horsepower'`
- **Q2**: `254`
- **Q3**: `With mean`
- **Q4**: `0`
- **Q5**: `0.029`
- **Q6**: `2.236`
